# U5-01 · Clases, archivos y el proyecto de la biblioteca

*Fundamentos de Programación en Python (IFCD0062) · Unidad 5*

Ejecuta cada celda con **Mayús + Intro**, en orden. Todo funciona en el navegador, sin instalar nada. Cambia el código y vuelve a ejecutarlo: es la mejor manera de aprender.

> Texto: *La vida de Lazarillo de Tormes y de sus fortunas y adversidades* (1554), edición digital del Proyecto Gutenberg (libro n.º 320), dominio público.

## 5.1 · Objetos y clases

`t1_clase.py`

In [ ]:
class Tratado:
    """Un capítulo del Lazarillo."""

    def __init__(self, numero, texto):
        self.numero = numero      # atributos del objeto
        self.texto = texto

    def palabras(self):           # un método
        return len(self.texto.split())

    def __str__(self):
        return f"Tratado {self.numero} ({self.palabras()} palabras)"

uno = Tratado(1, "Pues sepa V.M. ante todas cosas")
dos = Tratado(2, "Otro día, no pareciéndome estar allí seguro")
print(uno)
print(dos.numero, dos.palabras())
print(type(uno).__name__, isinstance(uno, Tratado))

`t2_libro.py`

In [ ]:
class Tratado:
    def __init__(self, nombre, texto):
        self.nombre, self.texto = nombre, texto
    def palabras(self):
        return len(self.texto.split())

class Libro:
    def __init__(self, ruta):
        with open(ruta, encoding="utf-8") as f:
            partes = f.read().split("Tratado ")
        self.tratados = []
        for parte in partes[1:]:
            nombre, _, texto = parte.partition("\n")
            self.tratados.append(Tratado(nombre.strip(), texto))
    def __len__(self):            # permite usar len(libro)
        return len(self.tratados)
    def mas_largo(self):
        return max(self.tratados, key=lambda t: t.palabras())

libro = Libro("lazarillo.txt")
t = libro.mas_largo()
print(len(libro), "tratados; el más largo:", t.nombre, t.palabras())

## 5.2 · Archivos

`t3_archivos.py`

In [ ]:
import csv
from pathlib import Path

texto = Path("lazarillo.txt").read_text(encoding="utf-8")
filas = []
for parte in texto.split("Tratado ")[1:]:
    nombre = parte.split()[0]
    filas.append({"tratado": nombre, "palabras": len(parte.split()),
                  "lineas": parte.count("\n")})
with open("resumen.csv", "w", newline="", encoding="utf-8") as f:
    campos = ["tratado", "palabras", "lineas"]
    w = csv.DictWriter(f, fieldnames=campos)
    w.writeheader()
    w.writerows(filas)
print(Path("resumen.csv").read_text(encoding="utf-8"))
with open("resumen.csv", encoding="utf-8") as f:
    total = sum(int(fila["palabras"]) for fila in csv.DictReader(f))
print("total:", total, "| existe:", Path("resumen.csv").exists())

`t6_json.py`

In [ ]:
import json

biblioteca = [
    {"titulo": "Lazarillo de Tormes", "autor": "Anónimo",
     "año": 1554, "leído": True},
    {"titulo": "La Celestina", "autor": "Fernando de Rojas",
     "año": 1499, "leído": False},
]
with open("biblioteca.json", "w", encoding="utf-8") as f:
    json.dump(biblioteca, f, ensure_ascii=False, indent=2)
with open("biblioteca.json", encoding="utf-8") as f:
    texto = f.read()
print("\n".join(texto.splitlines()[:7]))
datos = json.loads(texto)
print(type(datos).__name__, len(datos), datos[1]["leído"])

## 5.3 · Gestión de excepciones con objetos y archivos

`t4_errores.py`

In [ ]:
class FormatoNoValido(Exception):
    """Error propio: el archivo no tiene el formato esperado."""

def leer_numeros(ruta):
    try:
        with open(ruta, encoding="utf-8") as f:
            lineas = f.read().split()
    except FileNotFoundError:
        print(f"No existe {ruta}; se devuelve una lista vacía")
        return []
    try:
        return [float(x) for x in lineas]
    except ValueError as e:
        raise FormatoNoValido(f"{ruta}: {e}") from e

with open("bien.txt", "w") as f:
    f.write("3.5\n2\n10\n")
with open("mal.txt", "w") as f:
    f.write("3.5\ndos\n")
print(leer_numeros("bien.txt"))
print(leer_numeros("no_existe.txt"))
try:
    leer_numeros("mal.txt")
except FormatoNoValido as e:
    print("Error de formato ->", e)

## 5.4 · Un proyecto completo: una biblioteca personal

`t5_proyecto.py`

In [ ]:
import csv

class Libro:
    def __init__(self, titulo, autor, año):
        if not titulo:
            raise ValueError("el título no puede estar vacío")
        self.titulo, self.autor, self.año = titulo, autor, int(año)
    def __str__(self):
        return f"{self.titulo} ({self.autor}, {self.año})"

class Biblioteca:
    def __init__(self):
        self.libros = []
    def añadir(self, libro):
        self.libros.append(libro)
    def buscar(self, texto):
        t = texto.lower()
        return [l for l in self.libros
                if t in l.titulo.lower() or t in l.autor.lower()]
    def guardar(self, ruta):
        with open(ruta, "w", newline="", encoding="utf-8") as f:
            w = csv.writer(f)
            for l in self.libros:
                w.writerow([l.titulo, l.autor, l.año])
    @classmethod
    def cargar(cls, ruta):
        b = cls()
        with open(ruta, encoding="utf-8") as f:
            for titulo, autor, año in csv.reader(f):
                b.añadir(Libro(titulo, autor, año))
        return b

mia = Biblioteca()
mia.añadir(Libro("Lazarillo de Tormes", "Anónimo", 1554))
mia.añadir(Libro("La Celestina", "Fernando de Rojas", 1499))
mia.añadir(Libro("Novelas ejemplares", "Miguel de Cervantes", 1613))
mia.guardar("biblioteca.csv")
copia = Biblioteca.cargar("biblioteca.csv")
print(len(copia.libros), "libros cargados")
for libro in copia.buscar("de"):
    print(" -", libro)
try:
    copia.añadir(Libro("", "Nadie", 2026))
except ValueError as e:
    print("No se añade:", e)

**Prueba.** Añade a `Biblioteca` un método `__len__` y comprueba `len(copia)`.

**Ejercicio.** Haz que `Biblioteca` guarde y cargue los libros en JSON.